In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# PT-W3-D2：Ontology Compiler — 你已经写了 Semantic Compiler 的章程

> 📅 Week 3 - Day 2 | 2026-08-11 周二
>
> **并行轨道：Business Semantic Architecture**
>
> **今日主题：Ontology Compiler（语义编译器）**

---


## 一句话开篇

> **你以为自己在写 ADR，其实你在写编译器的 spec。**

你的 CRE BCM ADR-001 标题就叫"Business Composition Compiler principles charter"——这不仅是架构决策记录，它是一个 **Semantic Compiler（语义编译器）** 的设计章程。

今天我们要做的，是用 Ontology Compiler 视角重新读懂你已经写下的东西。

---


## 二、什么是 Ontology Compiler？

### 先用类比理解


In [ ]:
传统编译器：
  源代码（高级语言）
      ↓ 词法分析 / 语法分析 / 语义分析
  中间表示（IR）
      ↓ 代码生成
  机器码（可执行）

Ontology Compiler：
  业务语义（Ontology / Domain Model / BCM）
      ↓ Validate / Resolve / Assemble
  Design Artifact（语义中间表示）
      ↓ 下游产品物化
  Agent 可执行的能力组合（Skill / ApplicationContract）


### 为什么 Agent 需要 Compiler？

AI Agent 能直接读你的 Domain Model 吗？理论上能。但就像 CPU 能直接执行汇编一样——理论上可以，实际上没人愿意手写汇编。

Agent 需要的是：

1. **结构化的能力清单**（不是 307 张表的 DDL）
2. **显式的关系绑定**（不是外键 JOIN 推理）
3. **可推理的业务规则**（不是 if-else 代码考古）
4. **明确的执行约束**（不是"看审批流代码理解权限"）

Ontology Compiler 的任务就是把这些散落在代码、文档、ADR 中的业务语义，**编译**成一个 Agent 可以直接消费的结构化产物。

---


## 三、对照你的真实材料

### 你的 BCM ADR-001 已经定义了 Compiler 的三条根本原则

#### D-1：业务事实与运行时分离（Business ≠ Runtime）

> 矩阵及其伴生注册表只表达**稳定的业务语义**。运行时对象（BlueprintVersion / ApplicationContract / SkillRelease / DeploymentRevision）一律是**下游**。

**Ontology 视角翻译**：Ontology 描述"企业业务世界是什么"，不描述"系统怎么跑"。编译器的输入是业务语义，产物是可评审的 Design Artifact，**不是运行时对象**。

#### D-2：对象优先（Object First）

> 业务对象的类型（BusinessObjectType）与生命周期版本（LifecycleVersion）是首要锚点。状态归属在对象 + 生命周期版本上，不独立成顶层实体。

**Ontology 视角翻译**：Ontology 六维度里，Entity 和 Identity 是锚点。状态不是散落的字段，而是归属于 Entity 的生命周期阶段。编译器以对象为主键组织语义。

#### D-3：绑定优先（Binding First）

> 对象间的多对多关系以强类型、可版本化、可追溯的绑定类（Typed Binding）表达。

**Ontology 视角翻译**：Relationship 不是外键，是语义绑定。"Lease occupies Space"不是 `lease.space_id FK`，而是一个有类型、有版本、有约束的绑定声明。编译器消费这些绑定来组装 Agent 的认知地图。

---

### 你的 BCM ADR-005 定义了 Compiler 的五步流水线

这是真正令人兴奋的部分。你的 ADR-005 已经定义了一个**封闭且穷尽的五职责集合**：


In [ ]:
Validate → Resolve → Assemble → Emit → Report
  校验      解析       组装       产出     报告


让我们用 Ontology Compiler 视角逐一翻译：

#### 1. Validate（校验）— "语义合法性检查"

> 检查待编译要素的良构性与内部一致性：引用的对象、事件、能力、Skill、绑定是否都已知且形态合法。

**编译器视角**：这是语义编译的"类型检查（Type Check）"。

对应到 MI：如果你声明"Lease 终止时触发 Space 释放"，Validator 要检查：
- Lease 是否在 BusinessObjectType 注册表中？
- "终止"是否是 Lease 的合法 Lifecycle Transition？
- Space 释放的 Effect 是否在 effect-registry 中声明？
- 这个绑定是否在 Typed Binding 注册表中？

**关键洞察**：你的 ADR-006 四类对象关系 + effect-registry 5 类冻结效果 = Validator 的规则集。

#### 2. Resolve（解析）— "去引用化"

> 别名归一到 canonical capability 标识，解析 OwnershipAssertion，把触发事件引用绑定到事件类型。

**编译器视角**：这是"符号解析（Symbol Resolution）"。

对应到 MI：在代码里，同一个业务概念可能有多个名字——`shop_id` / `unit_id` / `space_id` 可能指向同一个物理空间。Resolver 把所有别名归一到 canonical identity：`Space(Building→Floor→Unit→Shop)`。

**关键洞察**：这正是 Ontology 的 Identity 维度——业务身份不是表 ID，而是语义层的一致标识。

#### 3. Assemble（组装）— "构建语义中间表示"

> 依据 Typed Binding 与声明式约束规则，把解析后的要素组装成一致的语义结构。

**编译器视角**：这是"生成中间表示（IR Generation）"。

对应到 MI：把"Lease 对象 + Lease→Space 绑定 + Termination Event + Release Space Effect + 招商审批 Policy"组装成一个**语义完整的能力单元**——Agent 读这一个单元，就理解了"退租"这件事的全貌。

**关键洞察**：组装的依据是你的 ADR-004 Typed Binding 类集（CapabilitySkillBinding / SkillTriggerBinding / SkillDeliverableBinding / RoleBlueprintSkillBinding）。

#### 4. Emit（产出）— "生成 Design Artifact"

> 把组装好的语义结构渲染成可评审、可版本化的 Design Artifact。

**编译器视角**：这是"生成目标文件（Code Generation）"。但产物不是可执行码，而是**可评审的语义文档**。

八字段语义结构：

| 字段 | 含义 |
|------|------|
| `goal` | 业务意图（一句话目标） |
| `capability_refs` | 引用的业务能力 |
| `trigger_event_refs` | 触发事件 |
| `input_artifact_refs` | 输入交付物 |
| `output_artifact_type_refs` | 输出交付物类型 |
| `skill_candidate_refs` | 候选 Skill |
| `governance` | 业务权威与约束 |
| `review_status` | 评审状态 |

**关键洞察**：这八字段就是 Agent 的"任务描述卡"。Agent 拿到一张 Design Artifact，就知道了：要做什么（goal）、能调用什么能力（capability_refs）、什么时候触发（trigger_event_refs）、需要什么输入（input_artifact_refs）、产出什么（output_artifact_type_refs）、有哪些约束（governance）。

#### 5. Report（报告）— "缺口暴露"

> 把校验缺口、未解析引用、弱证据呈报给评审账本。

**编译器视角**：这是"诊断信息（Diagnostics）"。

**关键洞察**：Compiler 发现"Lease 终止后 Inspection 的归属域未声明"时，不会假装知道答案，而是把缺口暴露出来。这正是 Ontology 的 Rule 维度——规则必须有显式来源，不编造。

---


## 四、完整编译流水线（用你的真实架构）


In [ ]:
你的已有资产                        Compiler 角色
─────────────────────────────────────────────────────────
MI Domain Model v1.0 (17 Context)   → Entity / Identity 输入
CRE BCM 14 域 + capability 行        → Capability 输入
ADR-006 四类对象关系                  → Relationship 输入
effect-registry.yaml (5 类 Effect)   → Event / Lifecycle 输入
business-ontology.yaml               → 模块→能力→场景 映射输入
审批流代码                            → Policy 输入（待显式化）
         │
         ▼
    ┌─────────────────────────┐
    │  Ontology Compiler       │
    │  Validate → Resolve →    │
    │  Assemble → Emit →       │
    │  Report                  │
    └─────────────────────────┘
         │
         ▼
    Design Artifact（八字段语义结构）
         │
         ▼
    LangChat Agent 消费
    → 组装成 ApplicationContract
    → 驱动 Digital Employee 执行


---


## 五、你的架构里已经实现了多少？

| Compiler 构件 | 你的已有资产 | 成熟度 | 缺口 |
|--------------|------------|--------|------|
| 输入：Entity | MI Domain Model §3 Object Ownership | ✅ 有 | 缺业务定义文本 |
| 输入：Relationship | ADR-006 四类关系 | ✅ 冻结 | 缺全部 Context 对的穷举 |
| 输入：Event | effect-registry 5 类 | ✅ 冻结 | 缺完整状态机 |
| 输入：Capability | CRE BCM capability 行 | ✅ 有 | 缺 Skill 映射 |
| 输入：Policy | 审批流代码 | ⚠️ 隐式 | 未显式声明 |
| 编译器章程 | BCM ADR-001 三原则 | ✅ published | 方向已冻结，未实现 |
| 编译器职责 | BCM ADR-005 五职责 | ✅ published | 6 域验证通过 |
| 编译器产物 | Design Artifact 八字段 | ✅ published | 物化形态待定（O-7） |
| 下游消费 | LangChat ADR-005..008 | ✅ published | 映射延后（O-8） |

**结论：你不需要"新建"一个 Ontology Compiler。你需要把已有的 BCM ADR-001 + ADR-005 从"架构决策"推进到"工程实现"。**

---


## 六、架构师视角

以前：Ontology Compiler 是一个我还没开始做的全新概念。

现在：我的 BCM ADR-001 就是 Ontology Compiler 的章程，ADR-005 是它的职责规范。我的 6 份 ADR + effect-registry + business-ontology.yaml 已经构成了编译器的**输入规范**。缺的不是设计，是把设计从"已确认方向"推进到"文档事实"的工程实现。

**这一步的关键认知转变**：


In [ ]:
以前：我在写架构决策记录（ADR）
现在：我在写 Semantic Compiler 的规范（spec）

以前：Design Artifact 是一个抽象概念
现在：Design Artifact 是 Agent 的"任务描述卡"——Agent 拿到它就知道能做什么、怎么做、受什么约束

以前：五职责（Validate/Resolve/Assemble/Emit/Report）是文档里的理论框架
现在：五职责是 Compiler 的流水线——和传统编译器的 Lexer/Parser/SemanticAnalyzer/Codegen/Diagnostics 一一对应


---


## 七、连接思考

本周主线 W11 是"Code Reality"——面对代码的真实状态。

今天的 Compiler 内容完美呼应：Ontology Compiler 的**输入**恰恰来自代码现实——307 张表的结构、effect-registry 的冻结效果、审批流的 if-else 逻辑。Compiler 不是空中楼阁，它是"把代码现实编译成 Agent 可消费的语义"的桥梁。

代码现实越清晰，Compiler 的输入越准确；Compiler 的产物越规范，Agent 的理解越可靠。

---


## 八、练习（5 分钟）

拿出你的 effect-registry.yaml，找到任意一条 Lifecycle Transition Effect（比如 `lease_terminated → release_space`）。

尝试手动走一遍五步编译：

1. **Validate**：这条 Effect 引用的对象和事件在注册表中存在吗？
2. **Resolve**：`release_space` 的 canonical capability 标识是什么？属于哪个 BCM 域？
3. **Assemble**：Lease 对象 + Termination Event + Release Effect + Inspection 约束 → 组装成一个什么语义结构？
4. **Emit**：填入八字段 Design Artifact——goal / capability_refs / trigger_event_refs / ...
5. **Report**：有没有发现缺口？（比如：Inspection 完成后的通知机制是否声明？）

> 如果你能走完这五步，你就亲手"编译"了第一个 Design Artifact。

---


## 九、明日预告

**PT-W3-D3：Ontology 如何约束 Agent 认知**

> Bounded Context = Agent 认知边界

你的 17 个 Bounded Context 不仅仅是代码组织方式——它们是 Agent 的"视野边界"。明天我们讨论：为什么 Agent 不应该"看到"所有表？Ontology 如何为 Agent 定义正确的认知范围？

---

*📁 本文保存于：/root/learning-notebooks/第11周/PT-W3-D2-Ontology-Compiler.md*
*🔗 关联主线：第11周 Code Reality*
*📚 并行轨道参考：parallel-track-enterprise-ontology.md §Week 3 Day 2*
